# RQ1 — Route–airline clustering

**Research question:** Do route–airline combinations with similar operational characteristics exhibit similar arrival-delay patterns?

We compare two ways of describing departure schedules: **Model A (evening-only)** and **Model B (morning / afternoon / evening-night)**. We cluster route–airline combinations *without using delay outcomes*, then compare their delay rates.

**Data:** shared 2024 cleaned Parquet file from `01c_cleaning_network.ipynb`. Run this notebook top to bottom after generating that file. Figures and tables are descriptive analyses of 2024, **not** out-of-sample EBM results.

## 1. Imports and project setup

In [ ]:
import os
import sys
from pathlib import Path

# Set before scikit-learn/joblib is imported, to avoid some Windows core-detection warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CLEAN_PARQUET, TABLES_DIR
from src.data_loading import load_flights

RANDOM_SEED = 42
MIN_FLIGHTS = 30
N_CLUSTERS = 5
KEYS = ["origin", "dest", "op_unique_carrier"]

## 2. Load cleaned flights and prepare operational features

We keep completed, non-diverted flights with positive distance and scheduled duration. We check HHMM clock validity and create time-of-day flags. The **three time shares sum to 1**; in Model B we use only two in the K-means distance to avoid exact redundancy. The models share the **same underlying observations**.

In [ ]:
df = load_flights(CLEAN_PARQUET)
print(f"Clean dataset: {len(df):,} rows")

flights = df.loc[
    (df["cancelled"] == 0)
    & (df["diverted"] == 0)
    & (df["distance"] > 0)
    & (df["crs_elapsed_time"] > 0)
].copy()

time = flights["crs_dep_time"]
valid_time = (
    time.notna()
    & time.between(0, 2400)
    & ((time % 100) < 60)
    & ((time < 2400) | (time == 2400))
)
flights = flights.loc[valid_time].copy()
flights["dep_hour"] = ((flights["crs_dep_time"] // 100) % 24).astype(int)

# Original definition: evening means 18:00–23:59
flights["evening"] = (flights["dep_hour"] >= 18).astype(int)
# Alternative three-period definition
flights["morning"] = flights["dep_hour"].between(5, 11).astype(int)
flights["afternoon"] = flights["dep_hour"].between(12, 17).astype(int)
flights["evening_night"] = ((flights["dep_hour"] >= 18) | (flights["dep_hour"] < 5)).astype(int)
flights["weekend"] = flights["day_of_week"].isin([6, 7]).astype(int)
flights["delayed_15"] = (flights["arr_delay"] >= 15).astype(int)

assert (flights[["morning", "afternoon", "evening_night"]].sum(axis=1) == 1).all()
print(f"Eligible completed flights: {len(flights):,}")

## 3. Aggregate to one row per route–airline combination

For each `(origin, destination, airline)` we calculate the number of flights, route distance and scheduling shares. We exclude combinations with fewer than 30 flights to avoid unstable proportions based on very few observations. **Arrival delays are not inputs to the clustering.**

In [ ]:
route_data_all = (
    flights.groupby(KEYS, observed=True)
    .agg(
        n_flights=("distance", "size"),
        mean_distance=("distance", "mean"),
        mean_duration=("crs_elapsed_time", "mean"),
        evening_share=("evening", "mean"),
        morning_share=("morning", "mean"),
        afternoon_share=("afternoon", "mean"),
        evening_night_share=("evening_night", "mean"),
        weekend_share=("weekend", "mean"),
    )
    .reset_index()
)
route_data = route_data_all.loc[route_data_all["n_flights"] >= MIN_FLIGHTS].copy()
print(f"Before threshold: {len(route_data_all):,} combinations")
print(f"After threshold: {len(route_data):,} combinations")
display(route_data.head(10))

## 4. Feature choice and correlation

Distance and scheduled duration describe similar aspects of route length. We inspect correlation and exclude scheduled duration from both main models to avoid effectively counting route length twice.

In [ ]:
corr = route_data[["mean_distance", "mean_duration", "n_flights", "evening_share", "weekend_share"]].corr()
display(corr.round(3))
print("Distance–duration correlation:", round(corr.loc["mean_distance", "mean_duration"], 3))

## 5. Compare two clustering feature sets

**Model A:** distance, yearly flights, evening share (18:00–23:59), weekend share.

**Model B:** distance, yearly flights, morning share (05:00–11:59), evening/night share (18:00–04:59), weekend share. Afternoon share is retained for describing clusters but omitted from K-means because the three shares sum to 1.

Both are standardized separately. Their internal quality scores describe their **own feature spaces** and are not a perfectly controlled head-to-head comparison: the evening/night definition also includes midnight–04:59, unlike Model A.

In [ ]:
FEATURES_A = ["mean_distance", "n_flights", "evening_share", "weekend_share"]
FEATURES_B = ["mean_distance", "n_flights", "morning_share", "evening_night_share", "weekend_share"]

scaler_a = StandardScaler()
scaler_b = StandardScaler()
X_a = scaler_a.fit_transform(route_data[FEATURES_A])
X_b = scaler_b.fit_transform(route_data[FEATURES_B])

def evaluate_k(X, model_name):
    rows = []
    for k in range(2, 11):
        labels = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit_predict(X)
        rows.append({
            "model": model_name,
            "k": k,
            "davies_bouldin": davies_bouldin_score(X, labels),
            "silhouette": silhouette_score(X, labels, sample_size=min(3000, len(X)), random_state=RANDOM_SEED),
        })
    return pd.DataFrame(rows)

evaluation_a = evaluate_k(X_a, "A_evening_only")
evaluation_b = evaluate_k(X_b, "B_time_of_day")
print("Model A — K evaluation")
display(evaluation_a.round(3))
print("Model B — K evaluation")
display(evaluation_b.round(3))

## 6. Fit K=5 models and interpret their operational characteristics

K=5 allows a direct descriptive comparison with our original analysis. It is **not necessarily optimal** for every possible clustering objective.

In [ ]:
model_a = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
model_b = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)

route_a = route_data.copy()
route_b = route_data.copy()
route_a["cluster"] = model_a.fit_predict(X_a)
route_b["cluster"] = model_b.fit_predict(X_b)

print("Model A: cluster sizes and characteristics")
display(route_a.groupby("cluster")[FEATURES_A].agg({"mean_distance":"mean", "n_flights":"mean", "evening_share":"mean", "weekend_share":"mean"}).round(3))
display(route_a["cluster"].value_counts().sort_index().rename("n_routes"))
print("Model B: cluster sizes and characteristics")
display(route_b.groupby("cluster")[["mean_distance", "n_flights", "morning_share", "afternoon_share", "evening_night_share", "weekend_share"]].mean().round(3))
display(route_b["cluster"].value_counts().sort_index().rename("n_routes"))

## 7. Compare arrival-delay behaviour **after** clustering

We define a late arrival as `arr_delay >= 15`. Each route–airline combination gets a delay rate. We report both the simple mean across routes and the flight-weighted rate; they answer different questions. We also compute eta-squared, the fraction of **route-level** delay-rate variation associated with cluster membership (descriptive and in-sample).

In [ ]:
delay_stats = (
    flights.groupby(KEYS, observed=True)
    .agg(
        delay_rate=("delayed_15", "mean"),
        mean_arr_delay=("arr_delay", "mean"),
        median_arr_delay=("arr_delay", "median"),
    )
    .reset_index()
)

def delay_summary(routes):
    merged = routes[KEYS + ["cluster", "n_flights"]].merge(
        delay_stats, on=KEYS, how="left", validate="one_to_one"
    )
    summary = merged.groupby("cluster").agg(
        n_routes=("delay_rate", "size"),
        avg_delay_rate=("delay_rate", "mean"),
        avg_arr_delay=("mean_arr_delay", "mean"),
        median_arr_delay=("median_arr_delay", "median"),
    )
    weighted = merged.groupby("cluster").apply(
        lambda g: np.average(g["delay_rate"], weights=g["n_flights"]),
        include_groups=False,
    )
    summary["weighted_delay_rate"] = weighted
    overall = merged["delay_rate"].mean()
    by_cluster = merged.groupby("cluster")["delay_rate"]
    ss_between = (by_cluster.size() * (by_cluster.mean() - overall) ** 2).sum()
    ss_total = ((merged["delay_rate"] - overall) ** 2).sum()
    eta_sq = ss_between / ss_total
    return merged, summary, eta_sq

route_with_delays_a, cluster_delays_a, eta_a = delay_summary(route_a)
route_with_delays_b, cluster_delays_b, eta_b = delay_summary(route_b)

print("Model A — delay patterns")
display(cluster_delays_a.round(3))
print(f"Model A eta-squared: {eta_a:.4f} ({100*eta_a:.2f}%)")
print("Model B — delay patterns")
display(cluster_delays_b.round(3))
print(f"Model B eta-squared: {eta_b:.4f} ({100*eta_b:.2f}%)")

## 8. Plot within-cluster variation

Boxes overlap if routes with similar operational features still have quite different delay outcomes. Cluster differences do not establish causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, merged, title in [
    (axes[0], route_with_delays_a, "Model A: evening-only"),
    (axes[1], route_with_delays_b, "Model B: morning/afternoon/evening-night"),
]:
    merged.boxplot(column="delay_rate", by="cluster", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Route-level delay rate")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 9. Robustness checks for the benchmark (Model A)

We evaluate whether assignments change with random initialization and with minimum flight counts of 30, 50 and 100. ARI=1 means identical assignments up to label names. Threshold ARIs compare only routes present in both datasets.

In [ ]:
seeds = [0, 1, 2, 3, 4, 42, 100, 123, 2024, 2026]
stability_df = pd.DataFrame([
    {
        "seed": seed,
        "ARI": adjusted_rand_score(route_a["cluster"], KMeans(n_clusters=N_CLUSTERS, random_state=seed, n_init=10).fit_predict(X_a)),
    }
    for seed in seeds
])
print("Model A initialization stability")
display(stability_df.round(4))
print("Mean ARI:", round(stability_df["ARI"].mean(), 4))

In [ ]:
threshold_rows = []
threshold_ari_rows = []
reference = route_a[KEYS + ["cluster"]].rename(columns={"cluster": "reference_cluster"})
for threshold in [30, 50, 100]:
    temp = route_data_all.loc[route_data_all["n_flights"] >= threshold].copy()
    X_temp = StandardScaler().fit_transform(temp[FEATURES_A])
    labels = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10).fit_predict(X_temp)
    temp["cluster"] = labels
    threshold_rows.append({
        "threshold": threshold,
        "n_combinations": len(temp),
        "davies_bouldin": davies_bouldin_score(X_temp, labels),
        "silhouette": silhouette_score(X_temp, labels, sample_size=min(3000, len(temp)), random_state=RANDOM_SEED),
    })
    shared = reference.merge(temp[KEYS + ["cluster"]], on=KEYS, how="inner", validate="one_to_one")
    threshold_ari_rows.append({
        "threshold": threshold,
        "shared_combinations": len(shared),
        "ARI_vs_30": adjusted_rand_score(shared["reference_cluster"], shared["cluster"]),
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_ari_df = pd.DataFrame(threshold_ari_rows)
display(threshold_df.round(3))
display(threshold_ari_df.round(3))

## 10. Results summary and takeaways

The cell below assembles an updated comparison from **this notebook run**, rather than relying on manually copied values. This makes the conclusion reproducible after rerunning on the shared cleaned dataset.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "A: evening only",
        "n_routes": len(route_a),
        "k": N_CLUSTERS,
        "davies_bouldin": davies_bouldin_score(X_a, route_a["cluster"]),
        "silhouette_sample": silhouette_score(X_a, route_a["cluster"], sample_size=min(3000, len(X_a)), random_state=RANDOM_SEED),
        "eta_squared": eta_a,
        "eta_squared_pct": 100 * eta_a,
    },
    {
        "model": "B: three departure periods",
        "n_routes": len(route_b),
        "k": N_CLUSTERS,
        "davies_bouldin": davies_bouldin_score(X_b, route_b["cluster"]),
        "silhouette_sample": silhouette_score(X_b, route_b["cluster"], sample_size=min(3000, len(X_b)), random_state=RANDOM_SEED),
        "eta_squared": eta_b,
        "eta_squared_pct": 100 * eta_b,
    },
])
display(comparison.round(3))
print("TAKEAWAYS")
print("1. Both models form clusters using operational features, without using delays as K-means inputs.")
print("2. Model A and B use the same route-airline sample, but different scheduling representations.")
print(f"3. Model A: DB={comparison.loc[0, 'davies_bouldin']:.3f}, silhouette={comparison.loc[0, 'silhouette_sample']:.3f}, eta²={eta_a:.3f}.")
print(f"4. Model B: DB={comparison.loc[1, 'davies_bouldin']:.3f}, silhouette={comparison.loc[1, 'silhouette_sample']:.3f}, eta²={eta_b:.3f}.")
print("5. More distinct delay rates do not prove stronger operational clusters or better delay prediction.")
print("6. The associations are descriptive, based on the same year used for aggregation.")
print("7. RQ3 must construct clusters from training-period information only to prevent leakage.")

### Takeaway for the report (draft)

We aggregated completed 2024 flights into route–airline combinations and clustered them based on route distance, service frequency and scheduling patterns. We compared a simpler **evening-only** feature representation with a richer **morning / afternoon / evening-night** representation. The more detailed model can distinguish morning-heavy from afternoon-heavy operations; however, its clusters are less separated under the internal clustering metrics than the evening-only model in our earlier experiments. Both approaches reveal differences in arrival-delay rates across clusters, but delay distributions overlap substantially. The eta-squared comparison quantifies these **in-sample associations**, not causal effects or predictive accuracy. Final selection for RQ3 requires a training-period fit and evaluation on held-out flights.

**Reporting caution:** Keep the model definitions and time boundaries explicit. The richer model changes both feature number and treatment of overnight departures, so this is a sensitivity comparison rather than a perfect one-variable experiment.

## 11. Export descriptive results

Export **both** model assignments and summaries for comparison. These full-year labels are for RQ1 analysis only; do not use them directly as leakage-free test features in RQ3.

In [ ]:
output_dir = TABLES_DIR
output_dir.mkdir(parents=True, exist_ok=True)
route_a.to_csv(output_dir / "02_route_clusters.csv", index=False)  # Benchmark / original path
route_b.to_csv(output_dir / "02_route_clusters_time_of_day.csv", index=False)
cluster_delays_a.to_csv(output_dir / "02_cluster_delay_summary.csv")
cluster_delays_b.to_csv(output_dir / "02_cluster_delay_summary_time_of_day.csv")
evaluation_a.to_csv(output_dir / "02_evaluation_evening_only.csv", index=False)
evaluation_b.to_csv(output_dir / "02_evaluation_time_of_day.csv", index=False)
comparison.to_csv(output_dir / "02_model_comparison.csv", index=False)
stability_df.to_csv(output_dir / "02_clustering_stability.csv", index=False)
threshold_df.to_csv(output_dir / "02_threshold_sensitivity.csv", index=False)
threshold_ari_df.to_csv(output_dir / "02_threshold_ari.csv", index=False)
print("Saved RQ1 tables in:", output_dir)